# 附录：手搓一个最小 Transformer 聊天模型

目标：不用 `nn.Transformer`，自己写出 Transformer 最核心的几块：

```text
Embedding → Self-Attention → Multi-Head → FeedForward → Block → 逐字生成
```

训练后它可以回答一些简单问题，例如：

```text
你是谁 → 我是一个手搓出来的小 Transformer。
什么是 Transformer → Transformer 的核心是注意力机制。
```

注意：这是课堂演示模型，不是通用 ChatGPT。它只会训练数据附近的问题。

## 1. 准备训练数据

每条数据写成同一种格式：

```text
用户：问题
助手：回答<结束>
```

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

pairs = [
    ('你好', '你好呀，很高兴见到你！'),
    ('你是谁', '我是一个手搓出来的小 Transformer。'),
    ('你能做什么', '我可以演示 Transformer 如何生成简单回复。'),
    ('什么是 AI', 'AI 可以理解成一个会学习的函数。'),
    ('什么是 Transformer', 'Transformer 的核心是注意力机制。'),
    ('怎么学 AI', '先理解输入输出，再学习模型和训练。'),
    ('在吗', '在的，我们继续学习。'),
    ('谢谢', '不客气，很高兴帮到你。'),
    ('再见', '再见，下次继续学习。'),
    ('hello', 'Hi，很高兴见到你！'),
]

text = '\n'.join([f'用户：{q}\n助手：{a}<结束>' for q, a in pairs])
text = '\n'.join([text] * 10)  # 小数据重复几遍，让小模型更容易学会
print(text[:160])

用户：你好
助手：你好呀，很高兴见到你！<结束>
用户：你是谁
助手：我是一个手搓出来的小 Transformer。<结束>
用户：你能做什么
助手：我可以演示 Transformer 如何生成简单回复。<结束>
用户：什么是 AI
助手：AI 可以理解成一个会学习的函数。<结束>
用户：什么是 Transformer


## 2. 字符表：把文字变成数字

这个版本用字符级建模。每个汉字、字母、标点都变成一个编号。

In [2]:
chars = sorted(set(text)) + ['<未知>']
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
unk_id = stoi['<未知>']

def encode(s):
    return [stoi.get(ch, unk_id) for ch in s]

def decode(ids):
    return ''.join(itos[i] for i in ids if itos[i] != '<未知>')

data = torch.tensor(encode(text), dtype=torch.long)
vocab_size = len(chars)
block_size = 64

print('字符表大小:', vocab_size)
print('训练文本长度:', len(data))

字符表大小: 100
训练文本长度: 3259


## 3. 训练任务：预测下一个字

Transformer 训练时不是直接“背答案”，而是不断练习：

```text
看到前面一串字 → 预测下一个字
```

In [3]:
def get_batch(batch_size=32):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

x, y = get_batch(2)
print('输入:', x.shape)
print('目标:', y.shape)

输入: torch.Size([2, 64])
目标: torch.Size([2, 64])


## 4. 手搓 Self-Attention

Self-Attention 做的事：每个位置都去看前面的上下文，决定哪些字更重要。

这里有三个向量：

- `query`：我现在想找什么信息。
- `key`：每个位置有什么信息可被匹配。
- `value`：真正拿来汇总的内容。

下三角 mask 保证模型只能看过去，不能偷看未来。

In [4]:
class Head(nn.Module):
    def __init__(self, n_emb, head_size):
        super().__init__()
        self.key = nn.Linear(n_emb, head_size, bias=False)
        self.query = nn.Linear(n_emb, head_size, bias=False)
        self.value = nn.Linear(n_emb, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        return wei @ self.value(x)

## 5. Multi-Head、前馈层和 Transformer Block

一个 Head 只从一个角度看上下文。Multi-Head 就是多个 Head 一起看。

Transformer Block 的结构：

```text
x → Attention → 残差 → FeedForward → 残差
```

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_emb, n_head):
        super().__init__()
        head_size = n_emb // n_head
        self.heads = nn.ModuleList([Head(n_emb, head_size) for _ in range(n_head)])
        self.proj = nn.Linear(n_emb, n_emb)

    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1))

class FeedForward(nn.Module):
    def __init__(self, n_emb):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_emb, 4*n_emb),
            nn.ReLU(),
            nn.Linear(4*n_emb, n_emb),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_emb, n_head):
        super().__init__()
        self.sa = MultiHeadAttention(n_emb, n_head)
        self.ff = FeedForward(n_emb)
        self.ln1 = nn.LayerNorm(n_emb)
        self.ln2 = nn.LayerNorm(n_emb)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

## 6. 拼成一个 Tiny Transformer

这里就是一个极小版 GPT：

```text
字符编号 → token embedding + position embedding → 多个 Block → 预测下一个字符
```

In [6]:
class TinyTransformer(nn.Module):
    def __init__(self, n_emb=64, n_head=4, n_layer=2):
        super().__init__()
        self.token = nn.Embedding(vocab_size, n_emb)
        self.position = nn.Embedding(block_size, n_emb)
        self.blocks = nn.Sequential(*[Block(n_emb, n_head) for _ in range(n_layer)])
        self.ln = nn.LayerNorm(n_emb)
        self.head = nn.Linear(n_emb, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        x = self.token(idx) + self.position(torch.arange(T))
        x = self.blocks(x)
        return self.head(self.ln(x))

model = TinyTransformer()
print('参数量:', sum(p.numel() for p in model.parameters()))

参数量: 116708


## 7. 训练模型

训练约二十秒。损失下降，说明模型越来越会预测下一个字。

In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

for step in range(1001):
    xb, yb = get_batch()
    logits = model(xb)
    loss = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 250 == 0:
        print(step, round(loss.item(), 4))

0 4.7857


250 0.0502


500 0.0321


750 0.0246


1000 0.0285


## 8. 开始聊天

生成时给它一个开头：

```text
用户：你好
助手：
```

然后让模型一个字一个字预测下去。

In [8]:
@torch.no_grad()
def chat(message, max_new_tokens=70):
    prompt = f'用户：{message}\n助手：'
    idx = torch.tensor([encode(prompt)], dtype=torch.long)

    for _ in range(max_new_tokens):
        logits = model(idx[:, -block_size:])
        next_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
        idx = torch.cat([idx, next_id], dim=1)

        out = decode(idx[0].tolist())
        if '<结束>' in out:
            break

    return out.split('助手：')[-1].split('<结束>')[0]

for q in ['你好', '你是谁', '你能做什么', '什么是 Transformer', '怎么学 AI', '再见']:
    print(q, '=>', chat(q))

你好 => 你好呀，很高兴见到你！
你是谁 => 我是一个手搓出来的小 Transformer。
你能做什么 => 我可以演示 Transformer 如何生成简单回复。
什么是 Transformer => Transformer 的核心是注意力机制。
怎么学 AI => 先理解输入输出，再学习模型和训练。
再见 => 再见，下次继续学习。


## 9. 这和 GPT 的关系

这个模型很小，但核心结构已经是 GPT 的简化版：

- 都是预测下一个 token。
- 都有位置编码。
- 都有 masked self-attention。
- 都有 multi-head attention。
- 都有 feedforward 和残差连接。

真正的 GPT-2 / GPT-4 只是规模大得多：数据更多、层数更多、参数更多、训练更久。